# Passo 2 — Preparar os dados e ajustar a primeira regressão

**Projeto da disciplina de Aprendizado de Máquina — Qualidade de vinhos**

No [primeiro passo](passo_1_qualidade_vinhos.ipynb), conhecemos os arquivos, investigamos duplicatas e definimos o alvo. Agora vamos justificar uma limpeza, separar os conjuntos e treinar uma **regressão linear**.

Ao terminar, teremos material para as seções **1. Objetivo da modelagem**, **2. Dados utilizados** e um primeiro resultado para **3. Desempenho e comparação**.

**Como usar:** execute as células em ordem com `Shift + Enter`. Edite as respostas do grupo ao final. Este notebook carrega os CSVs de `projeto/dados/` e cria suas próprias variáveis; não depende da memória do notebook anterior.

**Objetivo desta etapa:** obter a primeira previsão e entender como ela foi construída. A comparação com um modelo de referência será feita no passo 3 do [guia](passo_a_passo.md).


## 2.1 — Retomar a pergunta e o plano

> Queremos prever a **nota de qualidade de uma amostra de vinho**, a partir de suas **11 medidas físico-químicas e do tipo do vinho**, e o modelo devolve **um número**.

A nota real se chama `quality`. Ela ficará em `y`. As informações usadas para prever ficarão em `X`.

Vamos seguir esta sequência:

1. Ler os dados e criar uma cópia sem duplicatas completas.
2. Verificar se ainda existem entradas iguais com notas diferentes.
3. Reservar **20% para o teste final**.
4. Dividir os 80% restantes em dados de ajuste e validação.
5. Preparar as entradas dentro de um `Pipeline` e ajustar a regressão.
6. Fazer previsões na validação, medir o erro e interpretar os coeficientes.

A expressão “ajustar uma reta” se estende aqui a **regressão linear múltipla**, pois usaremos mais de uma entrada. A previsão será uma soma ponderada dessas informações, acrescida de um ponto de partida chamado intercepto.


## 2.2 — Importar as ferramentas

Usaremos as mesmas ferramentas de tabelas e gráficos do primeiro passo, agora com os componentes de modelagem do `scikit-learn`.

Selecione o kernel Python `.venv` do repositório. Em outro computador, instale as dependências de `requirements.txt`.


In [4]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEMENTE = 42
ALVO = "quality"
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Ambiente pronto.")
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)


Ambiente pronto.
pandas: 3.0.5
scikit-learn: 1.9.1


## 2.3 — Carregar os dados originais

Os dois arquivos já estão na pasta `dados`, ao lado dos notebooks. Execute o Jupyter a partir da raiz do repositório ou da pasta `projeto`.

Vamos manter os nomes originais das colunas. Por exemplo, acidez fixa aparece no CSV como **`fixed acidity`**. A tradução pode aparecer nas explicações, mas a leitura deve respeitar os nomes do arquivo.

Esta célula também verifica se os dados continuam numéricos, completos e dentro da escala esperada para o alvo. Se algum arquivo tiver sido alterado, a mensagem ajudará a identificar o problema.


In [ ]:
COLUNAS_ORIGINAIS = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide",
    "density", "pH", "sulphates", "alcohol", "quality",
]

pasta_atual = Path.cwd()
PASTA_PROJETO = pasta_atual if pasta_atual.name == "projeto" else pasta_atual / "projeto"
PASTA_DADOS = PASTA_PROJETO / "dados"

arquivos = {
    "red": PASTA_DADOS / "winequality-red.csv",
    "white": PASTA_DADOS / "winequality-white.csv",
}
tabelas = {}
for tipo, caminho in arquivos.items():
    if not caminho.is_file():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}. "
            "Confira a pasta de trabalho ou execute o carregamento do passo 1."
        )

    tabela = pd.read_csv(caminho, sep=";")
    if tabela.columns.tolist() != COLUNAS_ORIGINAIS:
        raise ValueError(f"Colunas inesperadas em {caminho.name}. Confira os nomes e o separador ';'.")
    if tabela.empty:
        raise ValueError(f"O arquivo {caminho.name} está vazio.")

    # Erros de conversão e valores ausentes exigem investigação.
    tabela = tabela.apply(pd.to_numeric, errors="raise")
    if not np.isfinite(tabela.to_numpy(dtype=float)).all():
        raise ValueError(f"Há ausências ou valores infinitos em {caminho.name}.")
    if not tabela[ALVO].between(0, 10).all() or not tabela[ALVO].mod(1).eq(0).all():
        raise ValueError(f"As notas de {caminho.name} devem ser inteiras entre 0 e 10.")
    tabelas[tipo] = tabela

dados_brutos = pd.concat(
    [
        tabelas["red"].assign(wine_type="red"),
        tabelas["white"].assign(wine_type="white"),
    ],
    ignore_index=True,
)

resumo_arquivos = pd.DataFrame({
    "Tabela": ["Tintos (original)", "Brancos (original)", "Base combinada"],
    "Linhas": [len(tabelas["red"]), len(tabelas["white"]), len(dados_brutos)],
    "Colunas": [tabelas["red"].shape[1], tabelas["white"].shape[1], dados_brutos.shape[1]],
})
# display(resumo_arquivos.style.set_caption("Dados carregados").hide(axis="index"))

display(resumo_arquivos)
display(dados_brutos.head())


,Tabela,Linhas,Colunas
0,Tintos (original),1599,12
1,Brancos (original),4898,12
2,Base combinada,6497,13


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


## 2.4 — Tratar as duplicatas e registrar a decisão

**Decisão adotada neste experimento:** remover linhas integralmente idênticas, mantendo a primeira ocorrência. Isso inclui as medidas, o tipo e a nota.

Nos arquivos do enunciado, esperamos retirar 1.177 repetições de 6.497 linhas e ficar com 5.320. A célula calcula os números efetivamente encontrados.

**Justificativa:** se duas cópias da mesma amostra estiverem em partições diferentes, a avaliação poderá parecer melhor do que seria em casos novos.

**Limitação:** não temos identificador de amostra para provar que cada repetição é um erro. A remoção altera o peso de perfis repetidos na análise. Por isso, ela precisa ser declarada no relatório.

A variável `dados_brutos` e os CSVs continuam disponíveis. A limpeza é feita em uma cópia chamada `base`. Manteremos também os índices das linhas originais para acompanhar sua divisão.


In [ ]:
quantidade_duplicatas = int(dados_brutos.duplicated().sum())
base = dados_brutos.drop_duplicates(keep="first").copy()

resumo_limpeza = pd.DataFrame({
    "Tipo": ["Tinto", "Branco"],
    "Antes": [int(dados_brutos["wine_type"].eq(tipo).sum()) for tipo in ["red", "white"]],
    "Depois": [int(base["wine_type"].eq(tipo).sum()) for tipo in ["red", "white"]],
})
resumo_limpeza["Removidas"] = resumo_limpeza["Antes"] - resumo_limpeza["Depois"]
display(resumo_limpeza.style.set_caption("Efeito da remoção de duplicatas").hide(axis="index"))

print("Linhas originais:", len(dados_brutos))
print("Repetições removidas:", quantidade_duplicatas)
print("Linhas mantidas:", len(base))
print("Colunas descartadas nesta limpeza: nenhuma.")

assert len(dados_brutos) - len(base) == quantidade_duplicatas
assert not base.duplicated().any()


### Ainda existem perfis iguais com notas diferentes?

Duas linhas podem ter todas as **entradas iguais**, mas notas diferentes. Elas não são duplicatas completas.

Se isso ocorrer, uma separação aleatória por linha pode colocar o mesmo perfil nos dois lados. Este notebook interrompe o fluxo nesse caso: seria necessário decidir como tratar a situação e adotar uma divisão por grupos antes de seguir.

Nos arquivos atuais, essa verificação deve encontrar zero perfis com notas diferentes. Isso permite seguir com a divisão por linha após a limpeza.


In [ ]:
COLUNAS_ENTRADA = [coluna for coluna in base.columns if coluna != ALVO]

notas_por_perfil = base.groupby(COLUNAS_ENTRADA, dropna=False)[ALVO].nunique()
perfis_conflitantes = int(notas_por_perfil.gt(1).sum())
print("Perfis de entrada iguais com mais de uma nota:", perfis_conflitantes)

if perfis_conflitantes:
    raise ValueError(
        "Há perfis repetidos com notas diferentes. Revise a política de tratamento "
        "e faça uma divisão por grupos de entradas antes de treinar."
    )

assert not base[COLUNAS_ENTRADA].duplicated().any()
print("Verificação concluída: cada perfil de entrada aparece uma vez na base limpa.")


## 2.5 — Separar entradas e alvo

Agora criamos `X` e `y` **a partir da base limpa**.

- `X` terá 11 medidas numéricas e `wine_type`.
- `y` terá somente `quality`.
- `quality` não entra em `X`: na aplicação, essa será justamente a informação que queremos estimar.

O tipo do vinho é categórico. Vamos codificá-lo mais adiante, dentro do pipeline.


In [ ]:
X = base.drop(columns=ALVO).copy()
y = base[ALVO].copy()

COLUNAS_NUMERICAS = [coluna for coluna in COLUNAS_ORIGINAIS if coluna != ALVO]
COLUNAS_CATEGORICAS = ["wine_type"]

assert ALVO not in X.columns
assert X.index.equals(y.index)

print("Formato de X:", X.shape)
print("Formato de y:", y.shape)
print("Entradas numéricas:", COLUNAS_NUMERICAS)
print("Entrada categórica:", COLUNAS_CATEGORICAS)
print("Alvo:", y.name)

display(X.head(3))
display(y.head(3).to_frame())


## 2.6 — Reservar o teste final

Separamos **80% para desenvolvimento** e **20% para teste final**. Desenvolvimento significa os dados que podemos usar para ajustar modelos e escolher alternativas.

`random_state=42` torna a divisão reproduzível com a mesma base e ordem de linhas. `stratify=y` mantém aproximadamente a proporção de cada nota nas duas partes.

Vamos guardar `X_teste` e `y_teste` em memória, mas **não calcular previsões ou métricas com eles nesta etapa**. O teste final será usado no passo 5, após as escolhas de modelo e de regra de decisão.

É preciso manter esta limpeza, a ordem das linhas e a semente nas etapas seguintes para reproduzir a mesma reserva.


In [ ]:
X_desenvolvimento, X_teste, y_desenvolvimento, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEMENTE,
    stratify=y,
)

assert set(X_desenvolvimento.index).isdisjoint(X_teste.index)
assert len(X_desenvolvimento) + len(X_teste) == len(base)

reserva = pd.DataFrame({
    "Conjunto": ["Desenvolvimento", "Teste final reservado"],
    "Linhas": [len(X_desenvolvimento), len(X_teste)],
    "Uso nesta etapa": ["Ajuste e validação interna", "Nenhuma previsão ou métrica"],
})
display(reserva.style.set_caption("Primeira divisão da base limpa").hide(axis="index"))


## 2.7 — Separar ajuste e validação dentro do desenvolvimento

Para verificar uma primeira previsão sem usar o teste final, separamos **25% do desenvolvimento para validação**. Os outros 75% serão usados para ajustar o pipeline.

Com os dados atuais, a divisão completa fica:

| Conjunto | Parte da base limpa | Uso |
| --- | --- | --- |
| Ajuste | 60% | Aprender as transformações e a regressão. |
| Validação | 20% | Observar previsões em casos não usados no ajuste. |
| Teste final | 20% | Avaliação final em uma etapa posterior. |

Os 25% são aplicados aos 80% de desenvolvimento: `0,25 × 0,80 = 0,20`.

No passo 3, poderemos usar validação cruzada dentro de todo o desenvolvimento para comparar os modelos com maior estabilidade. O teste final continuará sendo o mesmo.


In [ ]:
X_ajuste, X_validacao, y_ajuste, y_validacao = train_test_split(
    X_desenvolvimento,
    y_desenvolvimento,
    test_size=0.25,
    random_state=SEMENTE,
    stratify=y_desenvolvimento,
)

particoes = {
    "Ajuste": set(X_ajuste.index),
    "Validação": set(X_validacao.index),
    "Teste final": set(X_teste.index),
}
assert particoes["Ajuste"].isdisjoint(particoes["Validação"])
assert particoes["Ajuste"].isdisjoint(particoes["Teste final"])
assert particoes["Validação"].isdisjoint(particoes["Teste final"])
assert set.union(*particoes.values()) == set(base.index)

divisao = pd.DataFrame({
    "Conjunto": list(particoes),
    "Linhas": [len(indices) for indices in particoes.values()],
})
divisao["Percentual da base limpa"] = (100 * divisao["Linhas"] / len(base)).round(1)
display(divisao.style.set_caption("Partições sem sobreposição").hide(axis="index"))

distribuicao_interna = pd.concat(
    [
        y_ajuste.value_counts(normalize=True).rename("Ajuste (%)"),
        y_validacao.value_counts(normalize=True).rename("Validação (%)"),
    ],
    axis=1,
).fillna(0).sort_index().mul(100).round(2)
display(distribuicao_interna)


## 2.8 — Preparar as entradas dentro do pipeline

As medidas químicas têm escalas diferentes. O `StandardScaler` transforma cada uma subtraindo a média e dividindo pelo desvio-padrão **aprendidos no ajuste**.

Para a regressão linear sem penalização, a padronização ajuda a comparar os coeficientes na escala de um desvio-padrão; ela não garante menor erro.

Para `wine_type`, usamos `OneHotEncoder` com uma categoria de referência:

- tinto (`red`) → 0;
- branco (`white`) → 1.

A referência é tinto. Não precisamos de duas colunas indicadoras junto com o intercepto.

O `ColumnTransformer` aplica cada transformação às colunas certas. O `Pipeline` encadeia preparação e regressão, de modo que o mesmo preparo será usado ao prever.

Nesta célula apenas definimos o pipeline. Ele ainda não aprendeu médias, desvios ou coeficientes.


In [ ]:
preparacao = ColumnTransformer(
    transformers=[
        ("numericas", StandardScaler(), COLUNAS_NUMERICAS),
        (
            "tipo",
            OneHotEncoder(
                categories=[["red", "white"]],
                drop="first",
                sparse_output=False,
                handle_unknown="error",
            ),
            COLUNAS_CATEGORICAS,
        ),
    ],
    remainder="drop",
)

modelo = Pipeline(
    steps=[
        ("preparacao", preparacao),
        ("regressao", LinearRegression()),
    ]
)

print("Etapas:", " → ".join(nome for nome, _ in modelo.steps))
print("Numéricas: padronizadas com estatísticas do ajuste.")
print("Tipo: indicador de branco, usando tinto como referência.")
print("Um tipo diferente de red/white será rejeitado para exigir revisão.")


## 2.9 — Ajustar a regressão e conferir o preparo

`fit(X_ajuste, y_ajuste)` aprende as transformações e os coeficientes usando somente o conjunto de ajuste. Validação e teste final não participam desse aprendizado.

Depois do ajuste, vamos inspecionar algumas médias e desvios guardados pelo padronizador. Eles devem corresponder ao conjunto de ajuste.

A regressão linear padrão encontra coeficientes que minimizam a soma dos **erros ao quadrado** no ajuste. Podemos avaliar o resultado com **MAE**, mesmo que essa não seja a função minimizada durante o treinamento.


In [ ]:
modelo.fit(X_ajuste, y_ajuste)

padronizador = modelo.named_steps["preparacao"].named_transformers_["numericas"]
estatisticas_aprendidas = pd.DataFrame({
    "Variável": COLUNAS_NUMERICAS,
    "Média aprendida": padronizador.mean_,
    "Desvio-padrão aprendido": padronizador.scale_,
})

assert np.allclose(
    padronizador.mean_,
    X_ajuste[COLUNAS_NUMERICAS].mean().to_numpy(),
)
assert int(padronizador.n_samples_seen_) == len(X_ajuste)

display(
    estatisticas_aprendidas.style
    .format({"Média aprendida": "{:.4f}", "Desvio-padrão aprendido": "{:.4f}"})
    .set_caption("Transformações aprendidas somente no ajuste")
    .hide(axis="index")
)
print("Amostras utilizadas para ajustar o pipeline:", len(X_ajuste))
print("Regressão ajustada. O teste final continua reservado.")


## 2.10 — Prever na validação e medir o primeiro erro

Agora usamos `predict(X_validacao)`. O pipeline aplica as transformações já aprendidas e produz uma nota numérica para cada amostra.

O **MAE** é a média de `abs(nota_real - nota_prevista)`. Sua unidade é **ponto de nota**. Por exemplo, prever 6,4 quando a nota é 7 produz erro absoluto de 0,6 ponto.

Também vamos mostrar o MAE no ajuste. Ele descreve como o modelo se sai nos exemplos em que aprendeu e costuma ser otimista. A validação é uma evidência mais relevante para casos novos.

**Não arredondamos as previsões para calcular o MAE.** O arredondamento das tabelas serve apenas para exibição.


In [ ]:
previsoes_validacao = modelo.predict(X_validacao)
previsoes_ajuste = modelo.predict(X_ajuste)

mae_ajuste = mean_absolute_error(y_ajuste, previsoes_ajuste)
mae_validacao = mean_absolute_error(y_validacao, previsoes_validacao)

metricas = pd.DataFrame({
    "Conjunto": ["Ajuste", "Validação"],
    "Amostras": [len(y_ajuste), len(y_validacao)],
    "MAE (pontos de nota)": [mae_ajuste, mae_validacao],
})
display(
    metricas.style
    .format({"MAE (pontos de nota)": "{:.3f}"})
    .set_caption("Primeiro resultado da regressão linear")
    .hide(axis="index")
)

previsoes = pd.DataFrame({
    "tipo": X_validacao["wine_type"],
    "nota_real": y_validacao,
    "nota_prevista": previsoes_validacao,
})
previsoes["erro_absoluto"] = (previsoes["nota_real"] - previsoes["nota_prevista"]).abs()
previsoes.index.name = "linha_original"
display(previsoes.head(10).round(3))

assert np.isfinite(previsoes_validacao).all()
assert np.isclose(previsoes["erro_absoluto"].mean(), mae_validacao)
print(f"Na validação, o erro absoluto médio foi de {mae_validacao:.3f} ponto(s) de nota.")
print("Ainda precisamos comparar com a referência para avaliar o ganho do modelo.")


## 2.11 — Olhar os erros

O gráfico da esquerda compara nota real e prevista. A linha tracejada representa previsões iguais às notas reais.

À direita, mostramos o MAE por nota real **na validação**, acompanhado da quantidade de amostras. Grupos pequenos têm estimativas instáveis; um erro alto em uma única amostra não basta para generalizar.

O resumo por nota pode mostrar erros que a média global esconde. Esta é uma primeira análise exploratória da validação, e não uma avaliação final do projeto.


In [ ]:
erros_por_nota = (
    previsoes.groupby("nota_real")
    .agg(
        amostras=("erro_absoluto", "size"),
        MAE=("erro_absoluto", "mean"),
        media_prevista=("nota_prevista", "mean"),
    )
)
display(erros_por_nota.round(3))

fig, eixos = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

eixos[0].scatter(
    previsoes["nota_real"], previsoes["nota_prevista"],
    alpha=0.20, s=18, color="#7D263C",
)
limite_inferior = min(previsoes["nota_real"].min(), previsoes["nota_prevista"].min()) - 0.3
limite_superior = max(previsoes["nota_real"].max(), previsoes["nota_prevista"].max()) + 0.3
eixos[0].plot(
    [limite_inferior, limite_superior],
    [limite_inferior, limite_superior],
    "--", color="#17324D", label="Previsão igual à nota real",
)
eixos[0].set_xlim(limite_inferior, limite_superior)
eixos[0].set_ylim(limite_inferior, limite_superior)
eixos[0].set_xlabel("Nota real")
eixos[0].set_ylabel("Nota prevista")
eixos[0].set_title("Previsões na validação")
eixos[0].legend(fontsize=8)

barras = eixos[1].bar(
    erros_por_nota.index.astype(str), erros_por_nota["MAE"], color="#B89945",
)
for barra, quantidade in zip(barras, erros_por_nota["amostras"]):
    eixos[1].text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 0.03,
        f"n={int(quantidade)}",
        ha="center", fontsize=8,
    )
eixos[1].set_ylim(0, erros_por_nota["MAE"].max() * 1.22 + 0.05)
eixos[1].set_xlabel("Nota real")
eixos[1].set_ylabel("MAE (pontos de nota)")
eixos[1].set_title("Erro por nota na validação")

plt.show()


**Como interpretar:** se as previsões se concentrarem nas notas intermediárias, o modelo pode estar subestimando notas altas e superestimando notas baixas. Confira se o gráfico e a tabela sustentam essa leitura.

Um MAE de meio ponto não significa que todas as previsões erram no máximo meio ponto. A tabela pode conter erros individuais bem maiores.

Não conclua que o modelo é bom apenas por olhar o número. A referência do passo 3 mostrará o que conseguimos ao prever sempre a mediana do treino.


## 2.12 — Entender os coeficientes e o intercepto

A previsão pode ser escrita como:

> **nota prevista = intercepto + soma dos coeficientes × entradas transformadas**

Para cada medida padronizada, o coeficiente indica a mudança na nota prevista associada a **um desvio-padrão a mais naquela medida**, mantendo as outras entradas fixas.

Para `wine_type_white`, o coeficiente compara branco (1) com tinto (0), mantendo as medidas fixas. Esse coeficiente binário não usa a mesma unidade dos coeficientes numéricos.

O intercepto é a previsão quando as entradas numéricas padronizadas são zero, ou seja, estão nas médias do ajuste, e o tipo é tinto. Essa combinação de médias pode não representar uma amostra real.

**Cuidado na interpretação:** coeficientes são associações condicionais do modelo. Variáveis químicas correlacionadas podem tornar seus valores e sinais instáveis. Um coeficiente grande não demonstra causa nem substitui uma análise de importância.


In [ ]:
nomes_transformados = (
    modelo.named_steps["preparacao"].get_feature_names_out()
)
regressao = modelo.named_steps["regressao"]

coeficientes = pd.DataFrame({
    "variavel": [
        nome.replace("numericas__", "").replace("tipo__", "")
        for nome in nomes_transformados
    ],
    "coeficiente": regressao.coef_,
})
coeficientes["magnitude"] = coeficientes["coeficiente"].abs()
coeficientes["interpretacao_da_entrada"] = [
    "+1 desvio-padrão" if nome.startswith("numericas__") else "Branco (1) versus tinto (0)"
    for nome in nomes_transformados
]

display(
    coeficientes.sort_values("magnitude", ascending=False).style
    .format({"coeficiente": "{:+.3f}", "magnitude": "{:.3f}"})
    .set_caption("Coeficientes aprendidos no ajuste")
    .hide(axis="index")
)
print(f"Intercepto: {regressao.intercept_:.3f}")

coeficientes_numericos = coeficientes.loc[
    coeficientes["variavel"].isin(COLUNAS_NUMERICAS)
].copy()
top3 = coeficientes_numericos.nlargest(3, "magnitude")
print("Três maiores coeficientes numéricos em valor absoluto:")
display(top3[["variavel", "coeficiente"]].round(3))

para_grafico = coeficientes_numericos.sort_values("coeficiente")
fig, eixo = plt.subplots(figsize=(9, 5), constrained_layout=True)
eixo.barh(
    para_grafico["variavel"],
    para_grafico["coeficiente"],
    color=["#7D263C" if valor < 0 else "#2F6B56" for valor in para_grafico["coeficiente"]],
)
eixo.axvline(0, color="#17324D", linewidth=1)
eixo.set_xlabel("Mudança na nota prevista por +1 desvio-padrão, demais entradas fixas")
eixo.set_title("Coeficientes das medidas numéricas")
plt.show()


### Conferir a conta em uma amostra

Vamos usar uma amostra da **validação** para enxergar a soma de cada contribuição. Não é uma nova amostra externa e não pertence ao teste final.

A soma do intercepto com as contribuições deve reproduzir `predict()`. Essa conferência ajuda a entender a regressão múltipla como uma conta, em vez de apenas uma chamada de função.


In [ ]:
amostra = X_validacao.iloc[[0]].copy()
entradas_transformadas = modelo.named_steps["preparacao"].transform(amostra)[0]

contribuicoes = pd.DataFrame({
    "variavel": coeficientes["variavel"],
    "entrada_transformada": entradas_transformadas,
    "coeficiente": regressao.coef_,
})
contribuicoes["contribuicao_na_nota"] = (
    contribuicoes["entrada_transformada"] * contribuicoes["coeficiente"]
)
previsao_manual = float(
    regressao.intercept_ + contribuicoes["contribuicao_na_nota"].sum()
)
previsao_pipeline = float(modelo.predict(amostra)[0])
nota_real_amostra = float(y_validacao.loc[amostra.index[0]])

assert np.isclose(previsao_manual, previsao_pipeline)

display(amostra)
display(contribuicoes.round(4))
print(f"Intercepto: {regressao.intercept_:.3f}")
print(f"Soma das contribuições: {contribuicoes['contribuicao_na_nota'].sum():.3f}")
print(f"Previsão pela conta: {previsao_manual:.3f}")
print(f"Previsão pelo pipeline: {previsao_pipeline:.3f}")
print(f"Nota real desta amostra de validação: {nota_real_amostra:.0f}")


## 2.13 — Registrar as decisões e o resultado

A tabela seguinte resume o experimento deste passo. Os valores são calculados a partir da execução.

A regressão linear pode produzir valores fora da escala de 0 a 10. Vamos verificar se isso ocorreu na validação, sem alterar as previsões. Um eventual corte ou arredondamento seria uma nova escolha e precisaria ser justificado e avaliado.

Os resultados deste notebook vêm de uma única divisão interna. Eles podem mudar com outra semente ou outro tratamento dos dados.


In [ ]:
fora_da_escala = int(
    ((previsoes_validacao < 0) | (previsoes_validacao > 10)).sum()
)

registro = pd.DataFrame([
    ["Objetivo", "Prever quality como número"],
    ["Linhas originais", len(dados_brutos)],
    ["Duplicatas completas removidas", quantidade_duplicatas],
    ["Linhas após limpeza", len(base)],
    ["Entradas do modelo", X.shape[1]],
    ["Perfis iguais com notas diferentes", perfis_conflitantes],
    ["Semente das duas divisões", SEMENTE],
    ["Linhas de ajuste", len(X_ajuste)],
    ["Linhas de validação", len(X_validacao)],
    ["Linhas de teste final reservado", len(X_teste)],
    ["Padronização", "Aprendida somente no ajuste"],
    ["Codificação do tipo", "Tinto = 0; branco = 1"],
    ["Modelo", "Regressão linear múltipla"],
    ["MAE no ajuste", f"{mae_ajuste:.3f} ponto(s) de nota"],
    ["MAE na validação", f"{mae_validacao:.3f} ponto(s) de nota"],
    ["Previsões de validação fora de 0–10", fora_da_escala],
    ["Teste final avaliado neste passo?", "Não"],
], columns=["Item", "Resultado"])

display(registro.style.set_caption("Registro do passo 2").hide(axis="index"))


## Respostas do grupo

Edite esta célula e substitua os campos entre colchetes.

**1. Qual tratamento foi aplicado às duplicatas? Quantas linhas foram removidas e mantidas?**

[Use o resumo da limpeza e justifique a escolha. Explique por que não podemos afirmar que toda repetição era um erro.]

**2. Qual é a diferença entre ajuste, validação e teste final?**

[Informe a quantidade de amostras de cada conjunto e o que podemos fazer com cada um nesta etapa.]

**3. Por que a padronização deve aprender apenas com os dados de ajuste?**

[Explique o risco de usar informações da validação ou do teste para preparar o modelo.]

**4. Como o tipo tinto/branco foi codificado e qual é a categoria de referência?**

[Explique o que significam os valores 0 e 1.]

**5. Qual foi o MAE na validação? Como você explicaria esse número a alguém que não conhece aprendizado de máquina?**

[Use pontos de nota. Não transforme o MAE em porcentagem de acertos nem em erro máximo.]

**6. Em quais três medidas numéricas aparecem os maiores coeficientes em valor absoluto?**

[Informe os nomes, sinais e valores. Interprete um deles na escala de um desvio-padrão, mantendo as outras entradas fixas.]

**7. O que o intercepto representa neste pipeline?**

[Relacione-o às médias numéricas do ajuste e ao tipo de referência.]

**8. O gráfico mostra algum padrão de erro? A quantidade de exemplos por nota permite conclusões firmes?**

[Cite evidências da validação e tenha cuidado com notas raras.]

**9. Já podemos afirmar que o modelo é útil? Qual comparação ainda falta?**

[Explique a necessidade de comparar com uma referência e de avaliar no teste reservado após as escolhas.]

**10. Por que o coeficiente de uma variável não prova que mudar essa variável causará uma nota diferente?**

[Distinga associação preditiva de causalidade e comente a relação entre as medidas químicas.]


## Checklist de conclusão

- [ ] Executei o notebook inteiro, em ordem, com um kernel novo.
- [ ] Sei justificar a remoção das duplicatas completas.
- [ ] Conferi a existência de entradas iguais com notas diferentes.
- [ ] Separei entradas e alvo sem colocar `quality` em `X`.
- [ ] Entendi as duas divisões e registrei a semente.
- [ ] Ajustei as transformações somente com `X_ajuste`.
- [ ] Produzi previsões na validação e interpretei o MAE.
- [ ] Consigo explicar três coeficientes e o intercepto.
- [ ] Conferi uma previsão pela soma das contribuições.
- [ ] Mantive o teste final sem previsões ou métricas.
- [ ] Preenchi as respostas do grupo e salvei o notebook.

**Próximo passo:** criar a referência que prevê a mediana e comparar seu MAE com o da regressão em validação cruzada, dentro dos dados de desenvolvimento. Consulte o [passo 3 do guia](passo_a_passo.md#passo-3--criar-uma-referência-e-escolher-como-medir-o-erro).

**Referências:** enunciado `Projeto_disciplina.pdf`; [UCI Wine Quality](https://doi.org/10.24432/C56S3T); arquivos locais `dados/winequality-red.csv` e `dados/winequality-white.csv`, obtidos dos endereços indicados no primeiro passo.

**Reprodução:** este notebook não salva um modelo pronto para implantação. As tabelas, figuras e explicações ficam nas saídas do próprio arquivo. Para repetir o experimento, use os mesmos CSVs, a mesma ordem de linhas, a limpeza descrita e `random_state=42` nas duas divisões.
